In [1]:
import pandas as pd
import numpy as np
from datetime import datetime

In [2]:
#pulling data
df = pd.read_parquet("data/clean_data.parquet")

In [3]:
#changing churn_type to int
df["churn_type"] = df["churn_type"].astype(int)

In [4]:
#weighted engagement score
df["engagement_score"] = (
    df["completion_rate"] * 0.4 +
    (df["avg_watch_hours_per_week"] / 20) * 0.4 +
    (df["number_of_logins_per_week"] / 20) * 0.2
)

In [5]:
#to track customer loyalty, long-turned customers tend to churn less
reference_date = pd.Timestamp(datetime.now().date())
df["tenure_days"] = (reference_date - df["signup_date"]).dt.days

In [6]:
#tracks who has been inactive for the last 30 days
df["is_inactive"] = (df["days_since_last_watch"] > 30).astype(int)

In [7]:
#billing history
df["high_risk_billing"] = (
    (df["price_increase_last_6m"] == True) & (df["payment_failures"] > 0)).astype(int)

In [8]:
#tracks if customer had more than 3 support tickets
df["high_support"] = (df["tickets_last_30d"] > 3).astype(int)

In [9]:
# Billing cycle as binary feature (yearly = 1, monthly = 0)
df["is_yearly"] = (df["billing_cycle"] == "yearly").astype(int)

In [10]:
# Effective price paid per month (yearly subscribers pay discounted rate)
df["effective_monthly_price"] = np.where(
    df["billing_cycle"] == "yearly",
    df["yearly_price"] / 12,
    df["monthly_price"]
)

In [11]:
#has the customer cancelled?
df["is_cancelled"] = df["cancelled"].astype(int)

In [12]:
# Days until next renewal (closer to 0 = renewal imminent)
df["days_until_renewal"] = (df["next_renewal_date"] - reference_date).dt.days.clip(lower=0)

In [13]:
#converts string into numbers, also dropping the string columns
df = pd.get_dummies(df, columns=["country", "plan"], drop_first=False)

In [14]:
#changing boolean types into int
df["price_increase_last_6m"] = df["price_increase_last_6m"].astype(int)

In [15]:
df = df.drop(columns=[
    "user_id", "signup_date", "start_date",
    "billing_cycle", "yearly_price", "cancelled",
    "cancellation_date", "last_renewal_date", "next_renewal_date"
], errors="ignore")

In [16]:
# sanity check - confirm no nulls slipped through
print("Null check after feature engineering:")
print(df.isnull().sum()[df.isnull().sum() > 0])
print("No nulls found" if df.isnull().sum().sum() == 0 else "")

Null check after feature engineering:
Series([], dtype: int64)
No nulls found


In [17]:
print("Shape after feature engineering:", df.shape)
print("\nNew features added:")
new_features = ["tenure_days", "is_inactive", "high_risk_billing", "high_support", "is_yearly", "effective_monthly_price", "is_cancelled", "days_until_renewal"]
print(df[new_features].describe().round(2))
print("\nFinal columns:")
print(list(df.columns))

Shape after feature engineering: (10000, 35)

New features added:
       tenure_days  is_inactive  high_risk_billing  high_support  is_yearly  \
count     10000.00     10000.00           10000.00      10000.00   10000.00   
mean        746.94         0.49               0.13          0.34       0.51   
std         434.09         0.50               0.34          0.47       0.50   
min           0.00         0.00               0.00          0.00       0.00   
25%         366.00         0.00               0.00          0.00       0.00   
50%         744.00         0.00               0.00          0.00       1.00   
75%        1128.00         1.00               0.00          1.00       1.00   
max        1500.00         1.00               1.00          1.00       1.00   

       effective_monthly_price  is_cancelled  days_until_renewal  
count                 10000.00      10000.00            10000.00  
mean                     14.14          0.15              101.97  
std                  

In [18]:
df.head()

,age,monthly_price,avg_watch_hours_per_week,number_of_logins_per_week,days_since_last_watch,completion_rate,tickets_last_30d,price_increase_last_6m,payment_failures,churn_type,...,country_Portugal,country_South Africa,country_South Korea,country_Spain,country_Thailand,country_UK,country_USA,plan_Basic,plan_Premium,plan_Standard
0,55,9.99,12.43,19,50,0.24,4,0,2,2,...,True,False,False,False,False,False,False,True,False,False
1,50,14.99,7.06,4,45,0.22,5,0,0,1,...,False,False,False,False,False,False,False,False,False,True
2,21,19.99,15.38,17,44,0.54,1,0,1,2,...,False,False,False,False,False,False,False,False,True,False
3,24,14.99,9.08,11,58,0.35,0,1,2,0,...,False,False,False,False,False,False,False,False,False,True
4,21,14.99,5.01,18,34,0.57,1,1,0,0,...,False,False,True,False,False,False,False,False,False,True


In [19]:
df.to_parquet("data/features.parquet", index=False)